# Plotting multiple cutouts from a catalogue

In [ ]:
# imports
import astropy.units as u
from galfind.Data import morgan_version_to_dir

from galfind import EAZY, Catalogue, Catalogue_Cutouts, EPOCHS_Selector

In [ ]:
survey = "JOF"
version = "v11"
instrument_names = ["NIRCam"]
aper_diams = [0.32] * u.arcsec
forced_phot_band = ["F277W", "F356W", "F444W"]
min_flux_pc_err = 10.
SED_fit_params_arr = [
    {"templates": "fsps_larson", "lowz_zmax": 4.0},
    {"templates": "fsps_larson", "lowz_zmax": 6.0},
    {"templates": "fsps_larson", "lowz_zmax": None}
]

JOF_cat = Catalogue.pipeline(
    survey,
    version,
    instrument_names = instrument_names,
    version_to_dir_dict = morgan_version_to_dir,
    aper_diams = aper_diams,
    forced_phot_band = forced_phot_band,
    min_flux_pc_err = min_flux_pc_err
)
# load sextractor half-light radii
JOF_cat.load_sextractor_Re()

# load EAZY SED fitting results
for SED_fit_params in SED_fit_params_arr:
    EAZY_fitter = EAZY(SED_fit_params)
    EAZY_fitter(JOF_cat, aper_diams[0], load_PDFs = True, load_SEDs = True, update = True)

print(JOF_cat)

In [ ]:
# perform EPOCHS selection
epochs_selector = EPOCHS_Selector(allow_lowz = False, unmasked_instruments = "NIRCam")
EPOCHS_JOF_cat = epochs_selector(JOF_cat, aper_diams[0], EAZY_fitter, return_copy = True)

## Example 1: Plotting a catalogue of cutouts in a single band

In [ ]:
f444w_cat_cutouts = Catalogue_Cutouts.from_cat_filt(EPOCHS_JOF_cat, "F444W", 0.96 * u.arcsec, overwrite = False)
print(f444w_cat_cutouts)

In [ ]:
f444w_cat_cutouts.plot()